# Pipeline completo de forecasting para a base 2 (DailyClimate)
 
 O script pode ser executado a partir da raiz ou da pasta `notebooks`:
 
 `python notebooks/base_02.py`
 
 Todas as decisões de treino e avaliação ficam aqui para que a base possa ser
 reutilizada pelos quatro modelos do trabalho.


In [ ]:
from __future__ import annotations

import json
import platform
import shutil
import sys
import time
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.statespace.sarimax import SARIMAX

In [ ]:
BASE_ID = "base_02"
DATASET_NAME = "DailyClimate"
RAW_NAME = "DailyClimate.csv"
TARGET = "meantemp"
EXTERNALS = ["humidity", "wind_speed", "meanpressure"]
RANDOM_STATE = 42
HORIZON = 1
ORIGIN_STEP = 7
TRAIN_MIN_OBS = 365
TEST_FRACTION = 0.20
SEASONAL_PERIOD = 7

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / BASE_ID
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / BASE_ID
RAW_FILE = DATA_DIR / RAW_NAME

## `mae`

Calcula MAE sem incluir valores ausentes.


In [ ]:
def mae(y_true: pd.Series | np.ndarray, y_pred: pd.Series | np.ndarray) -> float:
    true_values = np.asarray(y_true, dtype=float)
    predicted_values = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(true_values) & np.isfinite(predicted_values)
    return float(np.mean(np.abs(true_values[mask] - predicted_values[mask])))


## `prepare_directories`

Cria as pastas locais e preserva uma cópia do raw dentro da base.


In [ ]:
def prepare_directories() -> None:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    root_raw = PROJECT_ROOT / "data" / RAW_NAME
    if not RAW_FILE.exists() and root_raw.exists():
        shutil.copy2(root_raw, RAW_FILE)
    if not RAW_FILE.exists():
        raise FileNotFoundError(f"Raw não encontrado: {RAW_FILE}")


## `load_and_clean`

Carrega, ordena e valida a série diária da base 2.


In [ ]:
def load_and_clean() -> pd.DataFrame:
    frame = pd.read_csv(RAW_FILE)
    required = {"date", TARGET, *EXTERNALS}
    missing_columns = required.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"Colunas ausentes: {sorted(missing_columns)}")
    frame["date"] = pd.to_datetime(frame["date"], errors="coerce")
    frame = frame.dropna(subset=["date", TARGET]).sort_values("date")
    frame = frame.drop_duplicates(subset="date", keep="last")
    frame = frame.set_index("date")
    frame = frame[[TARGET, *EXTERNALS]].apply(pd.to_numeric, errors="coerce")
    frame = frame.dropna()
    frame.index.name = "date"
    frame.to_csv(DATA_DIR / f"cleaned_{BASE_ID}.csv")
    return frame


## `build_features`

Cria features fechadas no passado e calendário do próximo dia.
 
 As externas são sempre defasadas. Assim, a linha na origem `t` não usa
 observações posteriores a `t` para prever `t + 1`.


In [ ]:
def build_features(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    features = pd.DataFrame(index=frame.index)
    target_series = frame[TARGET]
    for lag in (1, 2, 3, 7, 14, 30):
        features[f"target_lag_{lag}"] = target_series.shift(lag)
    for column in EXTERNALS:
        for lag in (1, 2, 7):
            features[f"{column}_lag_{lag}"] = frame[column].shift(lag)
    shifted_target = target_series.shift(1)
    for window in (7, 14, 30):
        features[f"target_roll_mean_{window}"] = shifted_target.rolling(window).mean()
        features[f"target_roll_std_{window}"] = shifted_target.rolling(window).std()
    next_dates = pd.Series(features.index, index=features.index).shift(-1)
    features["next_day_of_week"] = next_dates.dt.dayofweek
    features["next_day_of_year"] = next_dates.dt.dayofyear
    features["next_month"] = next_dates.dt.month
    features["next_day_sin"] = np.sin(2 * np.pi * features["next_day_of_week"] / 7)
    features["next_day_cos"] = np.cos(2 * np.pi * features["next_day_of_week"] / 7)
    features["next_year_sin"] = np.sin(2 * np.pi * features["next_day_of_year"] / 365.25)
    features["next_year_cos"] = np.cos(2 * np.pi * features["next_day_of_year"] / 365.25)
    features["forecast_date"] = next_dates.values
    features["target_next"] = target_series.shift(-1)
    features = features.dropna(subset=["target_next"])
    feature_columns = [column for column in features.columns if column not in {"forecast_date", "target_next"}]
    features.to_csv(DATA_DIR / f"features_{BASE_ID}.csv")
    try:
        features.to_parquet(DATA_DIR / f"features_{BASE_ID}.parquet")
    except ImportError:
        pass
    return features, feature_columns


## `run_stl`

Executa STL e persiste componentes e força da sazonalidade.


In [ ]:
def run_stl(frame: pd.DataFrame) -> dict[str, float]:
    decomposition = STL(frame[TARGET], period=SEASONAL_PERIOD, robust=True).fit()
    components = pd.DataFrame(
        {"trend": decomposition.trend, "seasonal": decomposition.seasonal, "resid": decomposition.resid},
        index=frame.index,
    )
    components.to_csv(DATA_DIR / f"stl_{BASE_ID}.csv")
    seasonal_strength = max(
        0.0,
        1.0 - np.var(decomposition.resid) / np.var(decomposition.seasonal + decomposition.resid),
    )
    result = {"seasonal_period": SEASONAL_PERIOD, "seasonal_strength": float(seasonal_strength)}
    (DATA_DIR / f"stl_{BASE_ID}.json").write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


## `make_origins`

Define tuning interno e origens finais sem consultar o teste.


In [ ]:
def make_origins(frame: pd.DataFrame) -> tuple[list[pd.Timestamp], pd.Timestamp]:
    test_start_position = max(TRAIN_MIN_OBS, int(len(frame) * (1 - TEST_FRACTION)))
    if test_start_position >= len(frame) - HORIZON:
        raise ValueError("A série não possui observações suficientes para o teste.")
    test_start = frame.index[test_start_position]
    origins = list(frame.index[test_start_position::ORIGIN_STEP])
    origins = [origin for origin in origins if origin < frame.index[-HORIZON]]
    return origins, test_start


## `tune_tabular_models`

Seleciona RF e Elastic Net somente no período anterior ao teste.


In [ ]:
def tune_tabular_models(features: pd.DataFrame, feature_columns: list[str], test_start: pd.Timestamp) -> dict[str, Any]:
    tuning = features[features.index < test_start].dropna(subset=feature_columns)
    X = tuning[feature_columns]
    y = tuning["target_next"]
    splits = min(3, max(2, len(X) // 100))
    splitter = TimeSeriesSplit(n_splits=splits)
    rf_search = GridSearchCV(
        RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
        {"n_estimators": [100], "max_depth": [6, None], "min_samples_leaf": [1, 3]},
        scoring="neg_mean_absolute_error",
        cv=splitter,
        n_jobs=-1,
    )
    rf_search.fit(X, y)
    elastic_search = GridSearchCV(
        Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(max_iter=10000, random_state=RANDOM_STATE))]),
        {"model__alpha": [0.01, 0.1, 1.0], "model__l1_ratio": [0.2, 0.5, 0.8]},
        scoring="neg_mean_absolute_error",
        cv=splitter,
        n_jobs=-1,
    )
    elastic_search.fit(X, y)
    elastic_params = {
        "alpha": elastic_search.best_params_["model__alpha"],
        "l1_ratio": elastic_search.best_params_["model__l1_ratio"],
    }
    return {"random_forest": rf_search.best_params_, "elastic_net": elastic_params}


## `tune_sarimax`

Escolhe uma ordem SARIMAX por AIC no trecho de tuning.


In [ ]:
def tune_sarimax(frame: pd.DataFrame, test_start: pd.Timestamp) -> dict[str, Any]:
    tuning = frame[frame.index < test_start]
    exog = tuning[EXTERNALS].shift(1).dropna()
    endog = tuning[TARGET].loc[exog.index]
    candidates = [((1, 0, 0), (0, 0, 0, SEASONAL_PERIOD)), ((1, 0, 1), (0, 0, 0, SEASONAL_PERIOD))]
    best = None
    for order, seasonal_order in candidates:
        try:
            fitted = SARIMAX(endog, exog=exog, order=order, seasonal_order=seasonal_order, trend="c").fit(disp=False)
            candidate = {"order": order, "seasonal_order": seasonal_order, "aic": float(fitted.aic)}
            if best is None or candidate["aic"] < best["aic"]:
                best = candidate
        except Exception:
            continue
    if best is None:
        raise RuntimeError("Nenhuma configuração SARIMAX pôde ser ajustada.")
    return best


## `tune_holt_winters`

Escolhe HW por AIC no período de tuning.


In [ ]:
def tune_holt_winters(frame: pd.DataFrame, test_start: pd.Timestamp) -> dict[str, Any]:
    tuning = frame.loc[frame.index < test_start, TARGET]
    candidates = [("add", "add", True), ("add", None, True), (None, "add", False)]
    best = None
    for trend, seasonal, damped in candidates:
        try:
            fitted = ExponentialSmoothing(
                tuning, trend=trend, seasonal=seasonal, damped_trend=damped if trend else False,
                seasonal_periods=SEASONAL_PERIOD if seasonal else None, initialization_method="estimated",
            ).fit(optimized=True)
            candidate = {"trend": trend, "seasonal": seasonal, "damped_trend": damped if trend else False, "aic": float(fitted.aic)}
            if best is None or candidate["aic"] < best["aic"]:
                best = candidate
        except Exception:
            continue
    if best is None:
        raise RuntimeError("Nenhuma configuração Holt-Winters pôde ser ajustada.")
    return best


## `fit_predict_at_origin`

Ajusta um modelo com histórico até a origem e prevê o próximo ponto.


In [ ]:
def fit_predict_at_origin(
    model_name: str,
    frame: pd.DataFrame,
    features: pd.DataFrame,
    feature_columns: list[str],
    origin: pd.Timestamp,
    params: dict[str, Any],
) -> float:
    forecast_date = frame.index[frame.index.get_loc(origin) + HORIZON]
    if model_name == "holt_winters":
        model = ExponentialSmoothing(
            frame.loc[:origin, TARGET], trend=params["trend"], seasonal=params["seasonal"],
            damped_trend=params["damped_trend"], seasonal_periods=SEASONAL_PERIOD if params["seasonal"] else None,
            initialization_method="estimated",
        ).fit(optimized=True)
        return float(model.forecast(HORIZON).iloc[-1])
    if model_name == "sarimax":
        train = frame.loc[:origin]
        train_exog = train[EXTERNALS].shift(1).dropna()
        train_endog = train[TARGET].loc[train_exog.index]
        future_exog = frame.loc[[forecast_date], EXTERNALS].shift(1)
        future_exog.iloc[0] = frame.loc[origin, EXTERNALS].to_numpy()
        model = SARIMAX(train_endog, exog=train_exog, order=tuple(params["order"]), seasonal_order=tuple(params["seasonal_order"]), trend="c").fit(disp=False)
        return float(model.forecast(HORIZON, exog=future_exog).iloc[-1])
    train_features = features[(features.index < origin) & (features["forecast_date"] <= origin)].dropna(subset=feature_columns)
    test_features = features.loc[[origin], feature_columns]
    if model_name == "random_forest":
        model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
    else:
        model = Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(max_iter=10000, random_state=RANDOM_STATE, **params))])
    model.fit(train_features[feature_columns], train_features["target_next"])
    return float(model.predict(test_features)[0])


## `evaluate_models`

Executa o mesmo calendário walk-forward para os quatro modelos.


In [ ]:
def evaluate_models(frame: pd.DataFrame, features: pd.DataFrame, feature_columns: list[str], origins: list[pd.Timestamp], params: dict[str, Any]) -> dict[str, pd.DataFrame]:
    results: dict[str, pd.DataFrame] = {}
    for model_name in ("sarimax", "holt_winters", "random_forest", "elastic_net"):
        rows = []
        for origin in origins:
            forecast_date = frame.index[frame.index.get_loc(origin) + HORIZON]
            started = time.perf_counter()
            prediction = fit_predict_at_origin(model_name, frame, features, feature_columns, origin, params[model_name])
            rows.append({
                "origin_date": origin, "forecast_date": forecast_date, "y_true": frame.loc[forecast_date, TARGET],
                "y_pred": prediction, "horizon_step": HORIZON, "elapsed_seconds": time.perf_counter() - started,
            })
        result = pd.DataFrame(rows)
        result["residual"] = result["y_true"] - result["y_pred"]
        result.to_csv(DATA_DIR / f"predictions_{model_name}_{BASE_ID}.csv", index=False)
        result[["origin_date", "forecast_date", "residual"]].to_csv(DATA_DIR / f"residuals_{model_name}_{BASE_ID}.csv", index=False)
        results[model_name] = result
    return results


## `persist_evaluation`

Persiste métricas, Ljung-Box, hiperparâmetros e resumo do experimento.


In [ ]:
def persist_evaluation(results: dict[str, pd.DataFrame], params: dict[str, Any], stl_info: dict[str, float], test_start: pd.Timestamp) -> pd.DataFrame:
    mae_rows = [{"model": name, "mae": mae(result["y_true"], result["y_pred"])} for name, result in results.items()]
    mae_table = pd.DataFrame(mae_rows).sort_values("mae").reset_index(drop=True)
    mae_table.insert(0, "rank", np.arange(1, len(mae_table) + 1))
    mae_table.to_csv(DATA_DIR / f"mae_{BASE_ID}.csv", index=False)
    ljung_rows = []
    for name, result in results.items():
        lag = min(10, max(1, len(result) // 5))
        test = acorr_ljungbox(result["residual"], lags=[lag], return_df=True).iloc[0]
        ljung_rows.append({"model": name, "lag": lag, "lb_stat": float(test["lb_stat"]), "p_value": float(test["lb_pvalue"])})
    pd.DataFrame(ljung_rows).to_csv(DATA_DIR / f"ljung_box_{BASE_ID}.csv", index=False)
    metadata = {
        "base_id": BASE_ID, "dataset": DATASET_NAME, "target": TARGET, "horizon": HORIZON,
        "origin_step": ORIGIN_STEP, "train_min_obs": TRAIN_MIN_OBS, "test_start": str(test_start),
        "seasonal_period": SEASONAL_PERIOD, "random_state": RANDOM_STATE, "models": params,
        "stl": stl_info, "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
    }
    (DATA_DIR / f"hyperparams_{BASE_ID}.json").write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
    (ARTIFACT_DIR / f"{BASE_ID}_metadata.json").write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")
    return mae_table


## `save_feature_importance`

Treina RF e Elastic Net no tuning e salva importâncias interpretáveis.


In [ ]:
def save_feature_importance(features: pd.DataFrame, feature_columns: list[str], params: dict[str, Any], test_start: pd.Timestamp) -> None:
    tuning = features[features.index < test_start].dropna(subset=feature_columns)
    X, y = tuning[feature_columns], tuning["target_next"]
    rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params["random_forest"]).fit(X, y)
    elastic = Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(max_iter=10000, random_state=RANDOM_STATE, **params["elastic_net"]))]).fit(X, y)
    importance = pd.DataFrame({"feature": feature_columns, "random_forest": rf.feature_importances_, "elastic_net": elastic.named_steps["model"].coef_})
    importance.to_csv(DATA_DIR / f"feature_importance_{BASE_ID}.csv", index=False)
    joblib.dump(rf, ARTIFACT_DIR / "random_forest.pkl")
    joblib.dump(elastic, ARTIFACT_DIR / "elastic_net.pkl")


## `save_final_artifacts`

Ajusta e salva uma versão final de cada modelo até a última origem.


In [ ]:
def save_final_artifacts(frame: pd.DataFrame, features: pd.DataFrame, feature_columns: list[str], params: dict[str, Any], last_origin: pd.Timestamp) -> None:
    train = frame.loc[:last_origin]
    train_exog = train[EXTERNALS].shift(1).dropna()
    train_endog = train[TARGET].loc[train_exog.index]
    sarimax = SARIMAX(
        train_endog,
        exog=train_exog,
        order=tuple(params["sarimax"]["order"]),
        seasonal_order=tuple(params["sarimax"]["seasonal_order"]),
        trend="c",
    ).fit(disp=False)
    holt_winters = ExponentialSmoothing(
        train[TARGET],
        trend=params["holt_winters"]["trend"],
        seasonal=params["holt_winters"]["seasonal"],
        damped_trend=params["holt_winters"]["damped_trend"],
        seasonal_periods=SEASONAL_PERIOD if params["holt_winters"]["seasonal"] else None,
        initialization_method="estimated",
    ).fit(optimized=True)
    tabular_train = features[(features.index < last_origin) & (features["forecast_date"] <= last_origin)].dropna(subset=feature_columns)
    random_forest = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params["random_forest"]).fit(tabular_train[feature_columns], tabular_train["target_next"])
    elastic_net = Pipeline([("scaler", StandardScaler()), ("model", ElasticNet(max_iter=10000, random_state=RANDOM_STATE, **params["elastic_net"]))]).fit(tabular_train[feature_columns], tabular_train["target_next"])
    for name, model in {
        "sarimax": sarimax,
        "holt_winters": holt_winters,
        "random_forest": random_forest,
        "elastic_net": elastic_net,
    }.items():
        joblib.dump(model, ARTIFACT_DIR / f"{name}.pkl")
        metadata = {"base_id": BASE_ID, "model": name, "last_origin": str(last_origin), "params": params[name], "python": platform.python_version()}
        (ARTIFACT_DIR / f"{name}_meta.json").write_text(json.dumps(metadata, indent=2, default=str), encoding="utf-8")


## `run_pipeline`

Executa todas as etapas da base 2 e retorna o ranking final.


In [ ]:
def run_pipeline() -> pd.DataFrame:
    prepare_directories()
    frame = load_and_clean()
    features, feature_columns = build_features(frame)
    stl_info = run_stl(frame)
    origins, test_start = make_origins(frame)
    tabular_params = tune_tabular_models(features, feature_columns, test_start)
    params = {
        "random_forest": tabular_params["random_forest"],
        "elastic_net": tabular_params["elastic_net"],
        "sarimax": tune_sarimax(frame, test_start),
        "holt_winters": tune_holt_winters(frame, test_start),
    }
    results = evaluate_models(frame, features, feature_columns, origins, params)
    save_feature_importance(features, feature_columns, params, test_start)
    save_final_artifacts(frame, features, feature_columns, params, origins[-1])
    return persist_evaluation(results, params, stl_info, test_start)


In [ ]:
if __name__ == "__main__":
    ranking = run_pipeline()
    print(ranking.to_string(index=False))
